In [15]:
import pickle

In [16]:
with open('data/taxi/raw/dev.pkl', 'rb') as f:
    data = pickle.load(f)

In [17]:
print(data['dev'][0])
print(len(data['dev']))
print(data['dim_process'])

[{'idx_event': 1, 'type_event': 5, 'time_since_start': 0.0, 'time_since_last_event': 0.0}, {'idx_event': 2, 'type_event': 1, 'time_since_start': 0.4363888888888889, 'time_since_last_event': 0.4363888888888889}, {'idx_event': 3, 'type_event': 5, 'time_since_start': 1.1122222222222222, 'time_since_last_event': 0.6758333333333333}, {'idx_event': 4, 'type_event': 0, 'time_since_start': 1.4241666666666666, 'time_since_last_event': 0.31194444444444436}, {'idx_event': 5, 'type_event': 5, 'time_since_start': 1.8616666666666666, 'time_since_last_event': 0.4375}, {'idx_event': 6, 'type_event': 3, 'time_since_start': 2.2730555555555556, 'time_since_last_event': 0.411388888888889}, {'idx_event': 7, 'type_event': 8, 'time_since_start': 3.278888888888889, 'time_since_last_event': 1.0058333333333334}, {'idx_event': 8, 'type_event': 3, 'time_since_start': 3.4175, 'time_since_last_event': 0.13861111111111102}, {'idx_event': 9, 'type_event': 8, 'time_since_start': 3.460277777777778, 'time_since_last_eve

In [21]:
type_events = [entry['type_event'] for row in data['dev'] for entry in row]
max_val = max(type_events)
min_val = min(type_events)

print("最大值:", max_val)
print("最小值:", min_val)


最大值: 9
最小值: 0


In [4]:
from src.data.sequence import Sequence,EventSequence
import torch
def list_of_dicts_to_sequence(event_list):
    inter_times = [event['time_since_last_event'] for event in event_list]
    inter_times = torch.tensor(inter_times, dtype=torch.float32)
    arrival_times = [event['time_since_start'] for event in event_list]
    type_event = [event['type_event'] for event in event_list]
    type_event = torch.tensor(type_event, dtype=torch.long)
    return EventSequence(
        arrival_times=arrival_times,
        inter_times=inter_times,
        type_event=type_event
    )


In [5]:
data.keys()

dict_keys(['dim_process', 'dev'])

In [6]:
sequence_list = [list_of_dicts_to_sequence(s) for s in data["dev"]]

In [7]:
from src.data.batch import Batch

In [8]:
from src.data.tpp_dataset import TppDataset
ds = TppDataset(sequence_list)
loader = ds.get_dataloader(
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

In [9]:
for batch in loader:
    print(batch.keys())
    break

['arrival_times', 'inter_times', 'non_pad_mask', 'type_seq', 't_start', 't_end']


In [10]:
batch.type_seq[2]

tensor([ 5,  0,  5,  0,  5,  0,  5,  0,  5,  0,  5,  0,  5,  0,  5,  0,  5,  0,
         5,  0,  5,  0,  5,  0,  5,  0,  5,  0,  5,  0,  5,  0,  5,  0,  5,  0,
        10, 10])

In [11]:
batch.type_seq

tensor([[ 8,  3,  8,  ...,  3, 10, 10],
        [ 5,  3,  8,  ...,  1, 10, 10],
        [ 5,  0,  5,  ...,  0, 10, 10],
        ...,
        [ 8,  3,  8,  ...,  1, 10, 10],
        [ 8,  3,  8,  ...,  3,  8,  3],
        [ 8,  3,  8,  ...,  3,  8,  3]])

In [12]:
batch.arrival_times

tensor([[0.0000e+00, 2.4028e-01, 2.9944e-01,  ..., 8.7328e+00, 1.0000e+04,
         1.0000e+04],
        [0.0000e+00, 5.2389e-01, 3.0122e+00,  ..., 9.2972e+00, 1.0000e+04,
         1.0000e+04],
        [0.0000e+00, 1.3083e-01, 1.4417e-01,  ..., 8.5356e+00, 1.0000e+04,
         1.0000e+04],
        ...,
        [0.0000e+00, 3.8611e-02, 1.7028e-01,  ..., 6.6997e+00, 1.0000e+04,
         1.0000e+04],
        [0.0000e+00, 5.8333e-02, 2.6139e-01,  ..., 6.5306e+00, 6.5650e+00,
         6.6547e+00],
        [0.0000e+00, 2.3389e-01, 3.6167e-01,  ..., 7.4928e+00, 7.6561e+00,
         7.8142e+00]])

In [13]:
batch.inter_times

tensor([[0.0000e+00, 2.4028e-01, 5.9167e-02,  ..., 2.4417e-01, 1.0000e+04,
         1.0000e+04],
        [0.0000e+00, 5.2389e-01, 2.4883e+00,  ..., 3.6500e-01, 1.0000e+04,
         1.0000e+04],
        [0.0000e+00, 1.3083e-01, 1.3333e-02,  ..., 2.7917e-01, 1.0000e+04,
         1.0000e+04],
        ...,
        [0.0000e+00, 3.8611e-02, 1.3167e-01,  ..., 2.9472e-01, 1.0000e+04,
         1.0000e+04],
        [0.0000e+00, 5.8333e-02, 2.0306e-01,  ..., 2.0222e-01, 3.4444e-02,
         8.9722e-02],
        [0.0000e+00, 2.3389e-01, 1.2778e-01,  ..., 2.4139e-01, 1.6333e-01,
         1.5806e-01]])

In [14]:
from config.config_loader import load_args_from_yaml 
args = load_args_from_yaml("config/THP.yaml")
args.dataset = "taxi"
base_dir = f"data/{args.dataset}"

FileNotFoundError: [Errno 2] No such file or directory: 'config/THP.yaml'

In [ ]:
from src.data.preparation import prepare_data_tpp
df, train_loader, val_loader, test_loader,dataset = prepare_data_tpp(
    args,
    base_dir
)

In [ ]:
for batch in train_loader:
    print(f"arival_times: {batch.arrival_times}")
    print(f"inter_times: {batch.inter_times}")
    print(f"type_event: {batch.type_seq}")
    print(f"non_pad_mask: {batch.non_pad_mask}")
    print(f"t_start: {batch.t_start}")
    print(f"t_end: {batch.t_end}")  
    break

arival_times: tensor([[0.0000e+00, 1.9250e-01, 3.7361e-01,  ..., 5.9325e+00, 1.0000e+07,
         1.0000e+07],
        [0.0000e+00, 4.4944e-01, 5.1722e-01,  ..., 8.1878e+00, 9.2558e+00,
         9.8750e+00],
        [0.0000e+00, 9.1111e-02, 1.5083e-01,  ..., 8.6022e+00, 9.1594e+00,
         9.3744e+00],
        ...,
        [0.0000e+00, 2.4806e-01, 4.8333e-01,  ..., 8.3994e+00, 8.5561e+00,
         8.6558e+00],
        [0.0000e+00, 3.1000e-01, 9.5417e-01,  ..., 9.5492e+00, 9.6739e+00,
         9.7053e+00],
        [0.0000e+00, 6.2333e-01, 8.4750e-01,  ..., 5.8517e+00, 1.0000e+07,
         1.0000e+07]])
inter_times: tensor([[0.0000e+00, 1.9250e-01, 1.8111e-01,  ..., 1.3528e-01, 1.0000e+07,
         1.0000e+07],
        [0.0000e+00, 4.4944e-01, 6.7778e-02,  ..., 2.0750e-01, 1.0681e+00,
         6.1917e-01],
        [0.0000e+00, 9.1111e-02, 5.9722e-02,  ..., 2.3917e-01, 5.5722e-01,
         2.1500e-01],
        ...,
        [0.0000e+00, 2.4806e-01, 2.3528e-01,  ..., 1.1889e-01, 1.5667e-01

In [ ]:
batch[:,:-3]

EventBatch(
  arrival_times: [32, 35],
  inter_times: [32, 35],
  non_pad_mask: [32, 35],
  type_seq: [32, 35],
  t_start: [32],
  t_end: [32]
)

In [ ]:
for batch in train_loader:
   print(
    batch.inter_times.max().item(),
    batch.inter_times.min().item(),
    batch.inter_times.mean().item()
)

   

10000000.0 0.0 246710.703125
10000000.0 0.0 197368.625
10000000.0 0.0 164473.875
10000000.0 0.0 296052.8125
10000000.0 0.0 164473.875
10000000.0 0.0 98684.4140625
10000000.0 0.0 230263.375
10000000.0 0.0 345394.90625
10000000.0 0.0 328947.5625
10000000.0 0.0 263158.09375
10000000.0 0.0 296052.8125
10000000.0 0.0 279605.46875
10000000.0 0.0 213816.0
10000000.0 0.0 328947.53125
10000000.0 0.0 246710.734375
10000000.0 0.0 230263.375
10000000.0 0.0 180921.265625
10000000.0 0.0 263158.09375
10000000.0 0.0 164473.875
10000000.0 0.0 246710.734375
10000000.0 0.0 230263.34375
10000000.0 0.0 230263.34375
10000000.0 0.0 296052.84375
10000000.0 0.0 312500.1875
10000000.0 0.0 164473.875
10000000.0 0.0 296052.8125
10000000.0 0.0 263158.09375
10000000.0 0.0 345394.90625
10000000.0 0.0 230263.375
10000000.0 0.0 263158.09375
10000000.0 0.0 263158.09375
10000000.0 0.0 148026.546875
10000000.0 0.0 246710.703125
10000000.0 0.0 328947.59375
10000000.0 0.0 246710.734375
10000000.0 0.0 197368.625
10000000.0 

In [ ]:
batch.inter_times

tensor([[0.0000e+00, 1.5972e-01, 3.5611e-01, 9.9167e-02, 1.9722e-02, 6.2778e-02,
         3.7222e-02, 2.5000e-01, 3.5278e-02, 1.1389e-01, 2.9722e-02, 7.0417e-01,
         2.8089e+00, 4.2972e-01, 1.4806e-01, 3.2028e-01, 8.7611e-01, 4.2083e-01,
         2.5000e-02, 2.0417e-01, 1.1806e-01, 1.0583e-01, 5.3056e-02, 6.5556e-02,
         1.1389e-02, 2.0528e-01, 4.1361e-01, 7.3056e-02, 3.4444e-02, 4.7778e-02,
         4.7222e-02, 1.3056e-01, 1.2500e-02, 5.3056e-02, 2.6667e-02, 1.3250e-01,
         6.3889e-02, 3.4722e-01],
        [0.0000e+00, 1.1472e-01, 4.9056e-01, 4.8889e-02, 3.1194e-01, 5.4444e-02,
         1.5556e-02, 2.7611e-01, 1.0169e+00, 3.4194e-01, 1.0836e+00, 4.7250e-01,
         1.9889e-01, 1.2722e-01, 4.4583e-01, 4.4250e-01, 2.6944e+00, 4.1139e-01,
         7.5833e-02, 1.6861e-01, 1.7500e-02, 1.5694e-01, 3.7361e-01, 6.7222e-02,
         4.3167e-01, 2.6722e-01, 9.8056e-02, 9.3611e-02, 1.5833e-02, 1.5111e-01,
         1.5722e-01, 1.0278e-01, 1.6667e-02, 2.0778e-01, 1.8611e-02, 7.6111

In [ ]:
batch.inter_times[0].max().item()

2.8088889122009277

In [ ]:
time =batch.arrival_times[0]

In [ ]:
print(time)

tensor([0.0000, 0.1597, 0.5158, 0.6150, 0.6347, 0.6975, 0.7347, 0.9847, 1.0200,
        1.1339, 1.1636, 1.8678, 4.6767, 5.1064, 5.2544, 5.5747, 6.4508, 6.8717,
        6.8967, 7.1008, 7.2189, 7.3247, 7.3778, 7.4433, 7.4547, 7.6600, 8.0736,
        8.1467, 8.1811, 8.2289, 8.2761, 8.4067, 8.4192, 8.4722, 8.4989, 8.6314,
        8.6953, 9.0425])


In [ ]:
import src.data.constants as Constants
def get_non_pad_mask(seq):
    """ Get the non-padding positions. """

    assert seq.dim() == 2
    return seq.ne(Constants.PAD).type(torch.float).unsqueeze(-1)

def get_non_pad_mask_type(seq):
    """ Get the non-padding positions. """

    assert seq.dim() == 2
    non_pad_mask = seq.ne(Constants.PAD_TOKEN_ID).type(torch.float).unsqueeze(-1)
    return non_pad_mask

In [ ]:
batch.arrival_times[1] #==Constants.PAD

tensor([ 0.0000,  0.1147,  0.6053,  0.6542,  0.9661,  1.0206,  1.0361,  1.3122,
         2.3292,  2.6711,  3.7547,  4.2272,  4.4261,  4.5533,  4.9992,  5.4417,
         8.1361,  8.5475,  8.6233,  8.7919,  8.8094,  8.9664,  9.3400,  9.4072,
         9.8389, 10.1061, 10.2042, 10.2978, 10.3136, 10.4647, 10.6219, 10.7247,
        10.7414, 10.9492, 10.9678, 11.0439, 11.3778, 11.4667])

In [ ]:
for batch in train_loader:
    non_pad_mask=get_non_pad_mask(batch.arrival_times)
    non_pad_mask_type=get_non_pad_mask_type(batch.type_seq)
    print(torch.equal(non_pad_mask, non_pad_mask_type))
    diff_mask = (non_pad_mask != non_pad_mask_type)
    print("不一致的位置个数:", diff_mask.sum().item())
    if diff_mask.sum().item() > 0:
        indices = torch.nonzero(diff_mask, as_tuple=False)
        print("差异位置索引:", indices)
        break
    indices = torch.nonzero(diff_mask, as_tuple=False)
    print("差异位置索引:", indices)

True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3), dtype=torch.int64)
True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3), dtype=torch.int64)
True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3), dtype=torch.int64)
True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3), dtype=torch.int64)
True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3), dtype=torch.int64)
True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3), dtype=torch.int64)
True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3), dtype=torch.int64)
True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3), dtype=torch.int64)
True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3), dtype=torch.int64)
True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3), dtype=torch.int64)
True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3), dtype=torch.int64)
True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3), dtype=torch.int64)
True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3), dtype=torch.int64)
True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3), dtype=torch.int64)
True
不一致的位置个数: 0
差异位置索引: tensor([], size=(0, 3),

In [ ]:
non_pad_mask[0,20,0]
non_pad_mask_type[0,20,0]

tensor(1.)

In [ ]:
batch.arrival_times[0,20]

tensor(3.1122)

In [ ]:
batch.type_seq[0,20]

tensor(8)